In [1]:
!pip install ultralytics kaggle


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 22.3 MB/s eta 0:00:00


In [2]:
from google.colab import files
files.upload()


Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"tharanidharan23","key":"322d9327b0c1c361e16bf0427826c0d2"}'}

In [3]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json


In [4]:
!kaggle datasets download -d andrewmvd/helmet-detection


Dataset URL: https://www.kaggle.com/datasets/andrewmvd/helmet-detection
License(s): CC0-1.0
 79% 308M/391M [00:00<00:00, 877MB/s] 
100% 391M/391M [00:00<00:00, 622MB/s]


In [5]:
!unzip helmet-detection.zip -d helmet_dataset


Archive:  helmet-detection.zip
  inflating: helmet_dataset/annotations/BikesHelmets0.xml  
  inflating: helmet_dataset/annotations/BikesHelmets1.xml  
  inflating: helmet_dataset/annotations/BikesHelmets10.xml  
  inflating: helmet_dataset/annotations/BikesHelmets100.xml  
  inflating: helmet_dataset/annotations/BikesHelmets101.xml  
  inflating: helmet_dataset/annotations/BikesHelmets102.xml  
  inflating: helmet_dataset/annotations/BikesHelmets103.xml  
  inflating: helmet_dataset/annotations/BikesHelmets104.xml  
  inflating: helmet_dataset/annotations/BikesHelmets105.xml  
  inflating: helmet_dataset/annotations/BikesHelmets106.xml  
  inflating: helmet_dataset/annotations/BikesHelmets107.xml  
  inflating: helmet_dataset/annotations/BikesHelmets108.xml  
  inflating: helmet_dataset/annotations/BikesHelmets109.xml  
  inflating: helmet_dataset/annotations/BikesHelmets11.xml  
  inflating: helmet_dataset/annotations/BikesHelmets110.xml  
  inflating: helmet_dataset/annotations/Bikes

In [6]:
import os
os.makedirs("dataset/images/train", exist_ok=True)
os.makedirs("dataset/images/val", exist_ok=True)
os.makedirs("dataset/labels/train", exist_ok=True)
os.makedirs("dataset/labels/val", exist_ok=True)


In [10]:
import os
import xml.etree.ElementTree as ET
import shutil
from sklearn.model_selection import train_test_split

# ✅ correct class labels from dataset
classes = {
    "With Helmet": 0,
    "Without Helmet": 1
}

# paths
img_dir = "helmet_dataset/images"
ann_dir = "helmet_dataset/annotations"

# list images
images = os.listdir(img_dir)

# train / validation split
train_imgs, val_imgs = train_test_split(images, test_size=0.2, random_state=42)

def convert(img_list, img_out, label_out):
    for img in img_list:
        xml_file = img.replace(".jpg", ".xml").replace(".png", ".xml")
        tree = ET.parse(os.path.join(ann_dir, xml_file))
        root = tree.getroot()

        w = int(root.find("size/width").text)
        h = int(root.find("size/height").text)

        label_path = os.path.join(
            label_out,
            img.replace(".jpg", ".txt").replace(".png", ".txt")
        )

        with open(label_path, "w") as f:
            for obj in root.findall("object"):
                cls_name = obj.find("name").text
                cls = classes[cls_name]

                box = obj.find("bndbox")
                xmin = int(box.find("xmin").text)
                ymin = int(box.find("ymin").text)
                xmax = int(box.find("xmax").text)
                ymax = int(box.find("ymax").text)

                x_center = ((xmin + xmax) / 2) / w
                y_center = ((ymin + ymax) / 2) / h
                bw = (xmax - xmin) / w
                bh = (ymax - ymin) / h

                f.write(f"{cls} {x_center} {y_center} {bw} {bh}\n")

        shutil.copy(os.path.join(img_dir, img), img_out)

# run conversion
convert(train_imgs, "dataset/images/train", "dataset/labels/train")
convert(val_imgs, "dataset/images/val", "dataset/labels/val")


In [11]:
%%writefile data.yaml
path: dataset
train: images/train
val: images/val

names:
  0: With Helmet
  1: Without Helmet


Writing data.yaml


In [12]:
!cat data.yaml


path: dataset
train: images/train
val: images/val

names:
  0: With Helmet
  1: Without Helmet


In [13]:
from ultralytics import YOLO

# load pretrained YOLOv8 nano model
model = YOLO("yolov8n.pt")

# train the model
model.train(
    data="data.yaml",
    epochs=30,
    imgsz=640,
    batch=16
)


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.11 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, ke

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7d39a8bad340>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.04804

In [14]:
from ultralytics import YOLO

# load trained model
model = YOLO("runs/detect/train/weights/best.pt")

# test on validation images
model.predict(
    source="dataset/images/val",
    save=True,
    conf=0.4
)



image 1/153 /content/dataset/images/val/BikesHelmets104.png: 448x640 2 With Helmets, 72.2ms
image 2/153 /content/dataset/images/val/BikesHelmets106.png: 448x640 2 With Helmets, 10.2ms
image 3/153 /content/dataset/images/val/BikesHelmets107.png: 192x640 1 With Helmet, 62.9ms
image 4/153 /content/dataset/images/val/BikesHelmets113.png: 480x640 1 With Helmet, 1 Without Helmet, 229.7ms
image 5/153 /content/dataset/images/val/BikesHelmets118.png: 576x640 1 With Helmet, 107.5ms
image 6/153 /content/dataset/images/val/BikesHelmets119.png: 448x640 1 With Helmet, 8.5ms
image 7/153 /content/dataset/images/val/BikesHelmets12.png: 480x640 1 With Helmet, 11.6ms
image 8/153 /content/dataset/images/val/BikesHelmets125.png: 480x640 2 With Helmets, 7.1ms
image 9/153 /content/dataset/images/val/BikesHelmets134.png: 416x640 6 With Helmets, 57.5ms
image 10/153 /content/dataset/images/val/BikesHelmets149.png: 480x640 2 With Helmets, 8.8ms
image 11/153 /content/dataset/images/val/BikesHelmets151.png: 384x6

[ultralytics.engine.results.Results object with attributes:
 
 boxes: ultralytics.engine.results.Boxes object
 keypoints: None
 masks: None
 names: {0: 'With Helmet', 1: 'Without Helmet'}
 obb: None
 orig_img: array([[[ 37,  43,  41],
         [ 37,  43,  41],
         [ 37,  43,  41],
         ...,
         [203, 214, 217],
         [194, 204, 208],
         [185, 196, 200]],
 
        [[ 36,  43,  40],
         [ 36,  43,  40],
         [ 36,  42,  40],
         ...,
         [200, 211, 214],
         [192, 203, 206],
         [183, 194, 197]],
 
        [[ 36,  42,  40],
         [ 36,  42,  40],
         [ 36,  43,  40],
         ...,
         [201, 212, 216],
         [197, 208, 212],
         [192, 203, 207]],
 
        ...,
 
        [[170, 165, 173],
         [171, 166, 175],
         [173, 168, 177],
         ...,
         [201, 198, 202],
         [201, 197, 201],
         [201, 197, 201]],
 
        [[170, 165, 173],
         [171, 166, 175],
         [173, 168, 177],
      

In [15]:
from google.colab import files
files.download("runs/detect/train/weights/best.pt")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>